# Половина 1: кросс-энкодер `mmBERT-base`

Первый из двух движков итогового бленда. Обучение с нуля, две стадии.

Половина сделана скриптовым конвейером: тетрадь только зовёт `src/scripts/*.py`
командными строками, вся логика — в них. Быстрый способ получить тело, точку
отсчёта на доске и локальную валидацию; вторая половина (`02_bge.ipynb`)
устроена наоборот.

    вход     формат k48/576 (48 ключей атрибутов, значение до 60 символов)
    доска    0.5419 в одиночку
    железо   RTX 5090 32 ГБ, 64 vCPU; стадия A 9.5 ч на эпоху, стадия B 25 мин

Метрика — macro PR-AUC по 20 категориям.

## 1. Окружение

Зависимости уже поставлены `bootstrap.ipynb`; она же закрепила `python3` и
`pip` за ядром Jupyter, поэтому все `!python3 ...` ниже идут тем же
интерпретатором. Ячейка ниже переводит рабочий каталог в корень репозитория и задаёт `ECUP_ROOT`:
все команды тетради зовут `src/scripts/...` от корня, а сама тетрадь лежит в
корне репозитория.

In [ ]:
import os
import pathlib
import sys

# Тетради лежат в корне репозитория — там же, где src/.
ROOT = pathlib.Path.cwd()
os.chdir(ROOT)
os.environ["ECUP_ROOT"] = str(ROOT)
os.environ["ECUP_N_JOBS"] = "16"
for d in ("/usr/local/bin", os.path.expanduser("~/.local/bin")):
    if os.path.isdir(d) and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = d + os.pathsep + os.environ["PATH"]
sys.path.insert(0, str(ROOT))
print("корень:", ROOT)

Артефакты складываются в `$ECUP_ROOT/artifacts`, данные — в `$ECUP_ROOT/data`.
Прогон воспроизводился на torch 2.11.0+cu128, transformers 5.5.3.

## 2. Данные

In [ ]:
!python3 -u src/scripts/01_fetch_data.py --group all

Четыре файла, 4.3 ГБ, около минуты.

## 3. Корпуса

Все в одном формате `--canon 0 --max-keys 48 --max-val-chars 60`.

In [ ]:
!python3 -u src/scripts/10_prep_text.py --split human --canon 0 --max-keys 48

!python3 -u src/scripts/10_prep_text.py --split llm --canon 0 --max-keys 48 --llm-pairs 800000 --out $ECUP_ROOT/artifacts/ce_text_llm_800k_raw_k48.parquet

!python3 -u src/scripts/13_prep_llm_chunked.py --chunks 6 --llm-pairs 11200000 --canon 0 --max-keys 48 --out $ECUP_ROOT/artifacts/ce_text_llm_11187k_raw_k48.parquet

!python3 -u src/scripts/07_noise_mask.py --text $ECUP_ROOT/artifacts/ce_text_human_raw_k48.parquet

!python3 -u src/scripts/05_build_index.py

Около 25 минут. `--canon 0` обязателен: по умолчанию канонизация включена и
текст получится другой. `05_build_index.py` строит общий индекс OOF — без него
стадия B падает на сохранении предсказаний.

## 4. Стадия A, две эпохи

**Два отдельных запуска**, а не один с `--epochs 2`: косинусное расписание
строится на всё число шагов сразу, поэтому в одном длинном прогоне шаг к концу
первой эпохи ещё высокий, и чекпойнт недоучен.

In [ ]:
!python3 -u src/scripts/20_pretrain.py --train-text $ECUP_ROOT/artifacts/ce_text_llm_11187k_raw_k48.parquet --val-text   $ECUP_ROOT/artifacts/ce_text_human_raw_k48.parquet --out $ECUP_ROOT/artifacts/mmb48_pre1 --model jhu-clsp/mmBERT-base --seed 101 --max-len 576 --batch 48 --lr 2e-5 --max-steps 233079 --val-fold 4 --eval-every 8000 --eval-max-pairs 20000 --save-every 2000 --workers 12 --bucket 1 --resume

!python3 -u src/scripts/20_pretrain.py --train-text $ECUP_ROOT/artifacts/ce_text_llm_11187k_raw_k48.parquet --val-text   $ECUP_ROOT/artifacts/ce_text_human_raw_k48.parquet --out $ECUP_ROOT/artifacts/mmb48_pre2 --model $ECUP_ROOT/artifacts/mmb48_pre1_last --seed 202 --max-len 576 --batch 48 --lr 1e-5 --max-steps 233079 --val-fold 4 --eval-every 8000 --eval-max-pairs 20000 --save-every 2000 --workers 12 --bucket 1 --resume

`--max-steps` = 11 187 780 / `--batch`, ровно одна эпоха. Меняете батч —
пересчитывайте.

В работу идут **последние** веса, `<out>_last`; в `<out>` лежит максимум
промежуточной валидации, он только для сравнения.

Вторая эпоха даёт +0.0022 доски за 9.5 часа.

## 5. Контекст каталога

In [ ]:
!python3 -u src/scripts/08_context_feats.py --text $ECUP_ROOT/artifacts/ce_text_human_raw_k48.parquet --out  $ECUP_ROOT/artifacts/ce_text_human_raw_k48_ctx.parquet

Около трёх часов на процессоре. Признаки нужны **только при обучении**: на
инференсе контекст не вычисляется.

## 6. Стадия B

Одна эпоха на шаге 2e-5. Две эпохи с пониженным шагом замерены и хуже.

In [ ]:
!python3 -u src/scripts/30_finetune.py --text $ECUP_ROOT/artifacts/ce_text_human_raw_k48_ctx.parquet --init-from $ECUP_ROOT/artifacts/mmb48_pre2_last --tag k48b1 --folds 0 --epochs 1 --lr 2e-5 --batch 48 --max-len 576 --swap-aug 1 --seed 1000 --workers 12 --cat-batches 1 --bootstrap --allow-text-change --grad-ckpt --replay $ECUP_ROOT/artifacts/ce_text_llm_800k_raw_k48.parquet --replay-share 0.2 --noise-mask $ECUP_ROOT/artifacts/ce_text_human_raw_k48.noise_0.8_0.3_0.8.npz --noise-weight 0.1

Реплей обязан быть в том же формате `k48`, что основной корпус.

**Про вес глушения.** Значение 0.1 проверено перебором: семейства 0.0, 0.05 и
0.1 по трём сидам каждое дают 0.7353 / 0.7367 / 0.7353 и статистически
неразличимы, а 0.2 и 0.3 заметно хуже (0.7328 и 0.7276). Глушить надо сильно,
конкретная величина в диапазоне до 0.1 роли не играет.

## 7. Архив половины 1 и стенд

Шаг обязательный: `03_blend.ipynb` достаёт из этого архива движок `src.ecup` и
чекпойнт `ce_f0` — только так ансамбль встречается с половиной 1.

`50_selftest.py` — стенд с настоящими лимитами: распаковывает архив, делает
каталог решения доступным только на чтение, глушит сеть и гоняет `run.py` на
трёх этапах (1000 пар / 60 с, 115 тыс. / 360 с, 275 тыс. / 780 с). Запас меньше
15% он считает провалом: на проверочной машине такой запас не повторится.

In [ ]:
!python3 src/scripts/40_build_submission.py --tag k48b1 --note "mmBERT: обе стадии на k48/576, маска шума, стадия B одна эпоха" --ce $ECUP_ROOT/artifacts/k48b1_f0 --ce-batch 256
!python3 src/scripts/50_selftest.py --zip $ECUP_ROOT/submissions/submission_k48b1.zip

Путь к архиву стенду передавайте абсолютным: сборщик кладёт его в
`$ECUP_ROOT/submissions`, а стенд ищет от текущего каталога.

## 8. Оценка

In [ ]:
!python3 src/scripts/36_clean_metric.py

Фолд 0 без текстово-невыучиваемых пар. **Сырую метрику фолда 0 как критерий не
использовать**: на семи моделях с известным скором доски она расставляет верно
17 пар сравнений из 21, а метрика с поправками — 20 из 21 (spearman +0.82
против +0.96).

Разброс прогона от сида: sd 0.0014 (размах 0.0032) по четырём запускам
одного рецепта. Различия
меньше 0.005 на одиночных прогонах не отличимы от случайности.